# PhoBERT NER (SKILL / TIME / LOC) trên Google Colab

**Chưa kiểm chứng trên Colab** — các bước được viết dựa trên môi trường đã chạy được ở máy local (Windows, Python 3.12, CPU).
Chỗ dễ vỡ nhất: phiên bản torch có sẵn của Colab + `transformers==4.46.3`.

Chuẩn bị: nén cả thư mục `phobert_ner/` **trừ** `.venv/`, `vncorenlp/`, `outputs/` thành `phobert_ner.zip`
(cần có: `VPhoBertTagger/` đã sửa, `data_build/`, `data_dir/`, `train_*.sh`, `requirements_working.txt`).

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Đưa project lên Colab (chọn MỘT cách)

In [ ]:
# Cách A: upload phobert_ner.zip
from google.colab import files
up = files.upload()                      # chọn phobert_ner.zip
!unzip -q -o phobert_ner.zip -d /content/

In [ ]:
# Cách B: đã để thư mục trên Google Drive tại MyDrive/phobert_ner
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/phobert_ner /content/

In [ ]:
%cd /content/phobert_ner
!ls

## 2. Java (VnCoreNLP cần Java — chỉ dùng khi sinh lại dữ liệu hoặc predict câu thô)

In [ ]:
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version

## 3. Thư viện Python (GIỮ torch có sẵn của Colab — bản CUDA)

In [ ]:
!grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!cat /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt
# processor.py gọi DataFrame.fillna(method=...) — đã deprecated ở pandas 2.x, có thể bị gỡ ở pandas 3 (chưa kiểm chứng)
import pandas, subprocess
if int(pandas.__version__.split('.')[0]) >= 3:
    subprocess.run(['pip', '-q', 'install', 'pandas>=2.2,<3'], check=True)
    print('Đã hạ pandas xuống <3 -> Runtime > Restart session rồi chạy tiếp từ bước 1 (%cd)')


## 4. Tải VnCoreNLP vào ./vncorenlp (đường dẫn tuyệt đối)

In [ ]:
import os
VN = os.path.abspath('vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
!find {VN} -type f | xargs ls -la

## 5. (Tuỳ chọn) Sinh lại dữ liệu + kiểm tra + baseline
Dữ liệu trong `data_dir/` đã có sẵn; chạy lại sẽ cho kết quả giống hệt (seed cố định).

In [ ]:
# !python data_build/build_data.py
!python data_build/validate_data.py
!python data_build/baseline_dict.py

## 6. Train
Mỗi script: train (early stop trên dev) → `--run_test` → dự đoán `test` + `test_hard` → báo cáo theo nhóm.
Kết quả trong `outputs/<arch>/`. **Chạy lần lượt, không song song** (dùng chung file cache trong `data_dir/`).

In [ ]:
!bash train_softmax.sh 2>&1 | grep -v 'it/s]'

In [ ]:
!bash train_crf.sh 2>&1 | grep -v 'it/s]'

In [ ]:
!bash train_lstm_crf.sh 2>&1 | grep -v 'it/s]'

## 7. So sánh với baseline trên test_hard

In [ ]:
!for A in softmax crf lstm_crf; do echo "== $A"; head -12 outputs/$A/report_test_hard.txt; done
!python data_build/eval_report.py data_dir/preds/baseline_test_hard.txt data_dir/test_hard_meta.jsonl | head -12

## 8. Thử câu thô (repo tự tách từ bằng VnCoreNLP-1.1.1, phải chạy từ thư mục gốc project)

In [ ]:
!printf 'tôi giỏi tiếng anh, rảnh vào tối và ở Hải Châu
' | python VPhoBertTagger/main.py predict --model_path outputs/crf/best_model.pt 2>&1 | grep -v EOFError | tail -3

## 9. Lưu kết quả về Drive (mỗi best_model.pt ~515MB)

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/phobert_ner_outputs && cp -r outputs logs /content/drive/MyDrive/phobert_ner_outputs/